# 18yA — Final empirical synthesis and evidence registry

This stage consolidates the verified outputs from 18s through 18x without
refitting a forecast model, reselecting a calibration transform or
reoptimising a trading rule.

It verifies the complete release chain, freezes the final sample-flow,
forecast, probability and trading tables, and records the evidential
boundary for every dissertation-facing conclusion.

The resulting tables distinguish verified empirical findings from
limitations and deferred extensions. In particular, the ten-date internal
holdout is treated as a descriptive check, while June 2026 remains the
principal external out-of-time evaluation block.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / '.git').exists():
    raise RuntimeError(f'Run this notebook from the repository root, not {ROOT}')

UTC = timezone.utc
STEP = '18yA'
TOL = 1e-6

S_DIR = ROOT / 'data/processed/18s_expanded_march_june_canonical_sample'
TA_DIR = ROOT / 'data/processed/18tA_hko_publication_availability'
TB_DIR = ROOT / 'data/processed/18tB_admissible_historical_information_sets'
UA_DIR = ROOT / 'data/processed/18uA_chronological_partition_and_folds'
UB_DIR = ROOT / 'data/processed/18uB_model_specific_freeze_support'
VA_DIR = ROOT / 'data/processed/18vA_residual_model_design_and_features'
VB1_DIR = ROOT / 'data/processed/18vB1_baseline_gp_temporal_oof'
VB2_DIR = ROOT / 'data/processed/18vB2_tree_temporal_oof'
VC_DIR = ROOT / 'data/processed/18vC_common_support_scoring_and_selection'
VD_DIR = ROOT / 'data/processed/18vD_selected_models_blind_predictions'
WA_DIR = ROOT / 'data/processed/18wA_contract_probability_mapping'
WB_DIR = ROOT / 'data/processed/18wB_development_probability_calibration'
WC_DIR = ROOT / 'data/processed/18wC_blind_calibrated_probability_release'
WD_DIR = ROOT / 'data/processed/18wD_holdout_external_probability_evaluation'
XA_DIR = ROOT / 'data/processed/18xA_development_trading_selection'
XB_DIR = ROOT / 'data/processed/18xB_blind_trading_signal_release'
XC_DIR = ROOT / 'data/processed/18xC_controlled_unblinding_trading_simulation'
XD_DIR = ROOT / 'data/processed/18xD_trading_robustness_and_figures'

OUT_DIR = ROOT / 'data/processed/18yA_final_empirical_synthesis'
REPORT_DIR = ROOT / 'reports/18yA_final_empirical_synthesis'
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RELEASES = [
    ('18s', S_DIR, '18s_expanded_sample_summary.json', '18s_expanded_integrity_checks.csv', '18s_expanded_sample_issues.csv', '18s_expanded_sha256_manifest.csv'),
    ('18tA', TA_DIR, '18tA_summary.json', '18tA_integrity_checks.csv', '18tA_issues.csv', '18tA_sha256_manifest.csv'),
    ('18tB', TB_DIR, '18tB_summary.json', '18tB_integrity_checks.csv', '18tB_issues.csv', '18tB_sha256_manifest.csv'),
    ('18uA', UA_DIR, '18uA_summary.json', '18uA_integrity_checks.csv', '18uA_issues.csv', '18uA_sha256_manifest.csv'),
    ('18uB', UB_DIR, '18uB_summary.json', '18uB_integrity_checks.csv', '18uB_issues.csv', '18uB_sha256_manifest.csv'),
    ('18vA', VA_DIR, '18vA_summary.json', '18vA_integrity_checks.csv', '18vA_issues.csv', '18vA_sha256_manifest.csv'),
    ('18vB1', VB1_DIR, '18vB1_summary.json', '18vB1_integrity_checks.csv', '18vB1_issues.csv', '18vB1_sha256_manifest.csv'),
    ('18vB2', VB2_DIR, '18vB2_summary.json', '18vB2_integrity_checks.csv', '18vB2_issues.csv', '18vB2_sha256_manifest.csv'),
    ('18vC', VC_DIR, '18vC_summary.json', '18vC_integrity_checks.csv', '18vC_issues.csv', '18vC_sha256_manifest.csv'),
    ('18vD', VD_DIR, '18vD_summary.json', '18vD_integrity_checks.csv', '18vD_issues.csv', '18vD_sha256_manifest.csv'),
    ('18wA', WA_DIR, '18wA_summary.json', '18wA_integrity_checks.csv', '18wA_issues.csv', '18wA_sha256_manifest.csv'),
    ('18wB', WB_DIR, '18wB_summary.json', '18wB_integrity_checks.csv', '18wB_issues.csv', '18wB_sha256_manifest.csv'),
    ('18wC', WC_DIR, '18wC_summary.json', '18wC_integrity_checks.csv', '18wC_issues.csv', '18wC_sha256_manifest.csv'),
    ('18wD', WD_DIR, '18wD_summary.json', '18wD_integrity_checks.csv', '18wD_issues.csv', '18wD_sha256_manifest.csv'),
    ('18xA', XA_DIR, '18xA_summary.json', '18xA_integrity_checks.csv', '18xA_issues.csv', '18xA_sha256_manifest.csv'),
    ('18xB', XB_DIR, '18xB_summary.json', '18xB_integrity_checks.csv', '18xB_issues.csv', '18xB_sha256_manifest.csv'),
    ('18xC', XC_DIR, '18xC_summary.json', '18xC_integrity_checks.csv', '18xC_issues.csv', '18xC_sha256_manifest.csv'),
    ('18xD', XD_DIR, '18xD_summary.json', '18xD_integrity_checks.csv', '18xD_issues.csv', '18xD_sha256_manifest.csv'),
]

KEY_INPUTS = {
    'support_flow': S_DIR / '18s_expanded_support_flow_summary.csv',
    'weather_error': S_DIR / '18s_expanded_weather_error_summary.csv',
    'market_binary': S_DIR / '18s_expanded_market_binary_score_summary.csv',
    'market_categorical': S_DIR / '18s_expanded_market_categorical_score_summary.csv',
    'u_block_summary': UA_DIR / '18uA_block_summary.csv',
    'development_scores': VC_DIR / '18vC_candidate_development_scores.csv',
    'development_selection': VC_DIR / '18vC_selected_candidates.json',
    'calibration_parameters': WB_DIR / '18wB_selected_calibration_parameters.csv',
    'calibration_effect': WD_DIR / '18wD_calibration_effect_summary.csv',
    'probability_support': WD_DIR / '18wD_evaluation_support_summary.csv',
    'probability_exact_common': WD_DIR / '18wD_exact_common_score_summary.csv',
    'strategy_registry': XA_DIR / '18xA_selected_strategy_registry.csv',
    'trading_headline': XD_DIR / '18xD_headline_trading_summary.csv',
    'trading_bootstrap': XD_DIR / '18xD_bootstrap_summary.csv',
}

for _, directory, summary_name, checks_name, issues_name, manifest_name in RELEASES:
    for path in [directory / summary_name, directory / checks_name, directory / issues_name, directory / manifest_name]:
        if not path.is_file():
            raise FileNotFoundError(f'Required verified release file is missing: {path}')

for role, path in KEY_INPUTS.items():
    if not path.is_file():
        raise FileNotFoundError(f'Required synthesis input is missing ({role}): {path}')

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(series: pd.Series, *, name: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    parsed = (
        series.astype(str).str.strip().str.lower().map(
            {'true': True, 'false': False, '1': True, '0': False, 'yes': True, 'no': False}
        )
    )
    if parsed.isna().any():
        bad = series.loc[parsed.isna()].drop_duplicates().tolist()
        raise ValueError(f'Could not parse Boolean column {name}: {bad}')
    return parsed.astype(bool)


def verify_manifest(path: Path) -> int:
    manifest = pd.read_csv(path)
    failures: list[str] = []
    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path
        if not candidate.is_file():
            failures.append(f'MISSING: {row.path}')
            continue
        if sha256_file(candidate) != row.sha256:
            failures.append(f'HASH: {row.path}')
        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f'SIZE: {row.path}')
    if failures:
        raise AssertionError(f'Manifest verification failed for {path}:\n' + '\n'.join(failures))
    return len(manifest)


release_rows = []
release_summaries: dict[str, dict[str, Any]] = {}

for stage, directory, summary_name, checks_name, issues_name, manifest_name in RELEASES:
    summary_path = directory / summary_name
    checks_path = directory / checks_name
    issues_path = directory / issues_name
    manifest_path = directory / manifest_name

    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    checks = pd.read_csv(checks_path)
    issues = pd.read_csv(issues_path)
    passed = parse_bool(checks['passed'], name=f'{stage}.passed')
    manifest_entries = verify_manifest(manifest_path)

    if summary.get('verdict') != 'PASS':
        raise AssertionError(f'{stage} is not a PASS release.')
    if not passed.all():
        raise AssertionError(f'{stage} contains failed integrity checks.')
    if not issues.empty:
        raise AssertionError(f'{stage} contains {len(issues)} issue rows.')

    release_summaries[stage] = summary
    release_rows.append(
        {
            'stage': stage,
            'verdict': summary.get('verdict'),
            'generated_at_utc': summary.get('generated_at_utc'),
            'summary_path': str(summary_path.relative_to(ROOT)),
            'manifest_path': str(manifest_path.relative_to(ROOT)),
            'manifest_entries_verified': manifest_entries,
            'integrity_checks_passed': int(passed.sum()),
            'integrity_checks_total': int(len(checks)),
            'issue_rows': int(len(issues)),
            'summary_sha256': sha256_file(summary_path),
            'manifest_sha256': sha256_file(manifest_path),
        }
    )

release_registry = pd.DataFrame(release_rows)

if len(release_registry) != 18:
    raise AssertionError(f'Expected 18 upstream release stages, found {len(release_registry)}.')
if not release_registry['verdict'].eq('PASS').all():
    raise AssertionError('An upstream release is not PASS.')
if release_registry['issue_rows'].sum() != 0:
    raise AssertionError('An upstream release contains issue rows.')

print('Complete 18s–18x release-chain verification: PASS')
display(release_registry)

Complete 18s–18x release-chain verification: PASS
    stage  ...                                    manifest_sha256
0     18s  ...  7d0726dab03c413a76e21115ee4c7de90dbf292738efda...
1    18tA  ...  57a376fa60391f705ba24448ed12266482d1e10304387c...
2    18tB  ...  5ee9bec9aa91384a9fc6dd30cc65f0574bec1bd63d2f58...
3    18uA  ...  fb1c224991307a6196d8dba0b0ea12b02e35165822fbc6...
4    18uB  ...  19e88c6c2b436c116cb1e034e9b6e161aabea65eef59c5...
5    18vA  ...  199f7152680bd329d7ed53166abfce1acdd347e99bf223...
6   18vB1  ...  17bfbd0e8046ecdfed05104ddacd80c515fb8e907b4e7b...
7   18vB2  ...  f803d9101d7b10ff2a4c867ac482f7da4b788a6399f62c...
8    18vC  ...  f3ace1658759b682b1dffa1c2a306cf2f69e1923dbe135...
9    18vD  ...  f9c743fc406ccdfe26c857b5027d85655e59b8c49af458...
10   18wA  ...  ed77e67a76cbb9939df696aff6aad94fa335e23e5a0142...
11   18wB  ...  a7fd96ef9b1486464fb71f595e80340af462442c9cc20d...
12   18wC  ...  f6ce8fab225878757df53f82b004bf777fbbe1be48ea3b...
13   18wD  ...  c00ddf7304

In [3]:
support_flow = pd.read_csv(KEY_INPUTS['support_flow'])
weather_error = pd.read_csv(KEY_INPUTS['weather_error'])
market_binary = pd.read_csv(KEY_INPUTS['market_binary'])
market_categorical = pd.read_csv(KEY_INPUTS['market_categorical'])
block_summary = pd.read_csv(KEY_INPUTS['u_block_summary'])
development_scores = pd.read_csv(KEY_INPUTS['development_scores'])
development_selection = json.loads(KEY_INPUTS['development_selection'].read_text(encoding='utf-8'))
calibration_parameters = pd.read_csv(KEY_INPUTS['calibration_parameters'])
calibration_effect = pd.read_csv(KEY_INPUTS['calibration_effect'])
probability_support = pd.read_csv(KEY_INPUTS['probability_support'])
probability_exact = pd.read_csv(KEY_INPUTS['probability_exact_common'])
strategy_registry = pd.read_csv(KEY_INPUTS['strategy_registry'])
trading_headline = pd.read_csv(KEY_INPUTS['trading_headline'])
trading_bootstrap = pd.read_csv(KEY_INPUTS['trading_bootstrap'])

s_summary = release_summaries['18s']
tA_summary = release_summaries['18tA']
uA_summary = release_summaries['18uA']
uB_summary = release_summaries['18uB']
vC_summary = release_summaries['18vC']
wD_summary = release_summaries['18wD']

sample_flow_rows = [
    ('Certified settlement dates', s_summary['certified_dates'], 'settlement dates', '18s'),
    ('Certified contracts', s_summary['certified_contracts'], 'contracts', '18s'),
    ('Contract-decision candidates', s_summary['contract_decision_candidates'], 'contract-rule rows', '18s'),
    ('Market-price-ready rows', s_summary['market_ready_rows'], 'contract-rule rows', '18s'),
    ('Deterministic-weather-ready rows', s_summary['weather_ready_rows'], 'contract-rule rows', '18s'),
    ('Exact market-weather common support', s_summary['exact_common_support_rows'], 'contract-rule rows', '18s'),
    ('Complete common-support books', s_summary['complete_common_support_books'], 'date-rule books', '18s'),
    ('Complete-book contract rows', s_summary['complete_book_contract_rows'], 'contract-rule rows', '18s'),
    ('History-only warm-up', uA_summary['history_only_warmup_dates'], 'settlement dates', '18uA'),
    ('Development block', uA_summary['development_dates'], 'settlement dates', '18uA'),
    ('Development model-ready support', uA_summary['development_model_ready_date_rule_rows'], 'date-rule rows', '18uA'),
    ('Common OOF selection support', vC_summary['common_candidate_selection_rows'], 'date-rule rows', '18vC'),
    ('Common OOF selection dates', vC_summary['common_candidate_selection_dates'], 'settlement dates', '18vC'),
    ('Locked internal holdout', uA_summary['internal_holdout_dates'], 'settlement dates', '18uA'),
    ('Holdout model-ready support', uA_summary['internal_holdout_model_ready_date_rule_rows'], 'date-rule rows', '18uA'),
    ('June external block', uA_summary['external_test_dates'], 'settlement dates', '18uA'),
    ('June model-ready support', uA_summary['external_model_ready_date_rule_rows'], 'date-rule rows', '18uA'),
    ('Exact market-common holdout books', wD_summary['exact_common_holdout_books'], 'date-rule books', '18wD'),
    ('Exact market-common June books', wD_summary['exact_common_external_books'], 'date-rule books', '18wD'),
]
final_sample_flow = pd.DataFrame(sample_flow_rows, columns=['object', 'count', 'unit', 'source_stage'])
final_sample_flow['count'] = final_sample_flow['count'].astype(int)

rule_labels = {
    '24h_prior': '24 hours prior',
    '12h_prior': '12 hours prior',
    '6h_prior': '6 hours prior',
    'event_day_open': 'Event-day open',
}
block_labels = {
    'march_may_baseline': 'March–May baseline',
    'june_external_extension': 'June external extension',
    'ALL': 'March–June combined',
    'INTERNAL_HOLDOUT': 'Internal holdout',
    'EXTERNAL_TEST': 'June external',
}
candidate_labels = {
    'raw_deterministic': 'Raw deterministic',
    'pooled_mean_residual': 'Pooled mean residual',
    'rule_mean_residual': 'Rule-specific mean residual',
    'pooled_empirical_residual': 'Pooled empirical residual',
    'rule_empirical_residual': 'Rule-specific empirical residual',
    'gp_rbf_rule': 'RBF GP',
    'gp_matern32_rule': 'Matérn-3/2 GP',
    'catboost_quantile_pooled': 'Pooled CatBoost',
    'catboost_quantile_rule': 'Rule-specific CatBoost',
}

deterministic_diagnostics = weather_error.loc[weather_error['sample_block'].eq('ALL')].copy()
deterministic_diagnostics['sample_block_label'] = deterministic_diagnostics['sample_block'].map(block_labels)
deterministic_diagnostics['decision_rule_label'] = deterministic_diagnostics['decision_rule'].map(rule_labels)
deterministic_diagnostics = deterministic_diagnostics.sort_values('decision_rule_order').reset_index(drop=True)[
    ['sample_block', 'sample_block_label', 'decision_rule', 'decision_rule_label',
     'n_complete_books', 'mean_forecast_daily_max_c', 'mean_hko_daily_max_c',
     'mean_error_c', 'mae_c', 'rmse_c', 'underforecast_rate',
     'deterministic_exact_contract_hit_rate']
]

market_diagnostics = market_binary.merge(
    market_categorical,
    on=['sample_block', 'decision_rule', 'decision_rule_order'],
    how='inner',
    validate='one_to_one',
    suffixes=('_binary', '_categorical'),
)
market_diagnostics = market_diagnostics.loc[~market_diagnostics['sample_block'].eq('ALL')].copy()
market_diagnostics['sample_block_label'] = market_diagnostics['sample_block'].map(block_labels)
market_diagnostics['decision_rule_label'] = market_diagnostics['decision_rule'].map(rule_labels)

raw_crps = float(
    development_scores.loc[
        development_scores['candidate_id'].eq('raw_deterministic'),
        'date_balanced_mean_crps_c',
    ].iloc[0]
)
development_model_selection = development_scores.copy()
development_model_selection['candidate_label'] = development_model_selection['candidate_id'].map(candidate_labels)
development_model_selection['crps_reduction_vs_raw_pct'] = (
    100.0 * (raw_crps - development_model_selection['date_balanced_mean_crps_c']) / raw_crps
)
role_map: dict[str, list[str]] = {}
for role in ['selected_baseline', 'selected_gaussian_process', 'selected_tree', 'selected_overall']:
    role_map.setdefault(development_selection[role], []).append(role)
development_model_selection['selection_roles'] = development_model_selection['candidate_id'].map(
    lambda candidate: '|'.join(role_map.get(candidate, []))
)
development_model_selection['selected_family_representative'] = development_model_selection['selection_roles'].ne('')
development_model_selection = development_model_selection.sort_values('overall_rank').reset_index(drop=True)

calibration_summary = calibration_parameters.copy()
calibration_summary['candidate_label'] = calibration_summary['candidate_id'].map(candidate_labels)

probability_model = probability_exact.copy()
probability_model['evaluation_block_label'] = probability_model['evaluation_block'].map(block_labels)
probability_model['candidate_label'] = probability_model['candidate_id'].map(candidate_labels)
probability_model['method_label'] = (
    probability_model['candidate_label']
    + ' ('
    + probability_model['probability_variant'].str.title()
    + ')'
)
probability_model = probability_model.rename(
    columns={
        'model_date_balanced_categorical_log_score': 'categorical_log_score',
        'model_date_balanced_multiclass_brier': 'multiclass_brier',
    }
)
probability_model['method_type'] = 'WEATHER_MODEL'

market_rows = []
for block, group in probability_exact.groupby('evaluation_block'):
    first = group.iloc[0]
    market_rows.append(
        {
            'candidate_id': 'normalised_market',
            'probability_variant': 'NORMALISED',
            'evaluation_block': block,
            'exact_common_books': int(first['exact_common_books']),
            'exact_common_dates': int(first['exact_common_dates']),
            'categorical_log_score': float(first['market_date_balanced_normalised_categorical_log_score']),
            'multiclass_brier': float(first['market_date_balanced_normalised_multiclass_brier']),
            'model_minus_market_categorical_log': 0.0,
            'model_minus_market_multiclass_brier': 0.0,
            'mean_raw_market_book_sum': float(first['mean_raw_market_book_sum']),
            'evaluation_block_label': block_labels[block],
            'candidate_label': 'Normalised market',
            'method_label': 'Normalised market',
            'method_type': 'MARKET',
        }
    )
probability_market = pd.DataFrame(market_rows)

probability_evaluation_full = pd.concat(
    [
        probability_market,
        probability_model[
            ['candidate_id', 'probability_variant', 'evaluation_block', 'exact_common_books',
             'exact_common_dates', 'categorical_log_score', 'multiclass_brier',
             'model_minus_market_categorical_log', 'model_minus_market_multiclass_brier',
             'mean_raw_market_book_sum', 'evaluation_block_label', 'candidate_label',
             'method_label', 'method_type']
        ],
    ],
    ignore_index=True,
)

compact_method_order = {
    'Normalised market': 0,
    'Pooled empirical residual (Uncalibrated)': 1,
    'Pooled empirical residual (Calibrated)': 2,
    'Matérn-3/2 GP (Uncalibrated)': 3,
    'Matérn-3/2 GP (Calibrated)': 4,
    'Pooled CatBoost (Calibrated)': 5,
}
probability_evaluation_compact = probability_evaluation_full.loc[
    probability_evaluation_full['method_label'].isin(compact_method_order)
].copy()
probability_evaluation_compact['method_order'] = probability_evaluation_compact['method_label'].map(compact_method_order)
probability_evaluation_compact['block_order'] = probability_evaluation_compact['evaluation_block'].map(
    {'INTERNAL_HOLDOUT': 0, 'EXTERNAL_TEST': 1}
)
probability_evaluation_compact = probability_evaluation_compact.sort_values(
    ['block_order', 'method_order']
).reset_index(drop=True)

trading_strategy_selection = strategy_registry.copy()
trading_strategy_selection['candidate_label'] = trading_strategy_selection['candidate_id'].map(candidate_labels)
trading_strategy_selection['decision_rule_label'] = trading_strategy_selection['decision_rule'].map(rule_labels)

trading_evaluation = trading_headline.copy()
trading_evaluation['candidate_label'] = trading_evaluation['candidate_id'].map(candidate_labels)
trading_evaluation['evaluation_block_label'] = trading_evaluation['evaluation_block'].map(block_labels)
trading_evaluation['decision_rule_label'] = trading_evaluation['decision_rule'].map(rule_labels)
trading_evaluation['strategy_label'] = trading_evaluation['strategy_role'].map(
    {
        'PRIMARY_OVERALL': 'Pooled empirical',
        'GAUSSIAN_PROCESS_FAMILY': 'Matérn GP',
        'TREE_FAMILY': 'CatBoost',
    }
)
trading_evaluation = trading_evaluation.sort_values(
    ['strategy_role', 'evaluation_block']
).reset_index(drop=True)

print('Final synthesis tables constructed: PASS')
display(final_sample_flow)
display(development_model_selection)
display(probability_evaluation_compact)
display(trading_evaluation)

Final synthesis tables constructed: PASS
                                 object  count                unit source_stage
0            Certified settlement dates    103    settlement dates          18s
1                   Certified contracts   1133           contracts          18s
2          Contract-decision candidates   4532  contract-rule rows          18s
3               Market-price-ready rows   4186  contract-rule rows          18s
4      Deterministic-weather-ready rows   4125  contract-rule rows          18s
5   Exact market-weather common support   3889  contract-rule rows          18s
6         Complete common-support books    350     date-rule books          18s
7           Complete-book contract rows   3850  contract-rule rows          18s
8                  History-only warm-up     25    settlement dates         18uA
9                     Development block     38    settlement dates         18uA
10      Development model-ready support    144      date-rule rows         18uA

In [4]:
def value_at(frame: pd.DataFrame, filters: dict[str, Any], column: str) -> float:
    mask = pd.Series(True, index=frame.index)
    for key, value in filters.items():
        mask &= frame[key].eq(value)
    subset = frame.loc[mask, column]
    if len(subset) != 1:
        raise AssertionError(f'Expected one row for {filters}, found {len(subset)}.')
    return float(subset.iloc[0])


# Final freeze assertions against the approved release.
expected_counts = {
    'certified_dates': 103,
    'certified_contracts': 1133,
    'contract_decision_candidates': 4532,
    'market_ready_rows': 4186,
    'weather_ready_rows': 4125,
    'exact_common_support_rows': 3889,
    'complete_common_support_books': 350,
}
for key, expected in expected_counts.items():
    actual = int(s_summary[key])
    if actual != expected:
        raise AssertionError(f'{key}: expected {expected}, found {actual}.')

expected_selection = {
    'selected_baseline': 'pooled_empirical_residual',
    'selected_gaussian_process': 'gp_matern32_rule',
    'selected_tree': 'catboost_quantile_pooled',
    'selected_overall': 'pooled_empirical_residual',
}
for key, expected in expected_selection.items():
    if development_selection[key] != expected:
        raise AssertionError(f'{key}: expected {expected}, found {development_selection[key]}.')

expected_values = [
    (development_model_selection, {'candidate_id': 'pooled_empirical_residual'}, 'date_balanced_mean_crps_c', 0.7112529005084938),
    (development_model_selection, {'candidate_id': 'gp_matern32_rule'}, 'date_balanced_mean_crps_c', 0.7400407887834148),
    (development_model_selection, {'candidate_id': 'catboost_quantile_pooled'}, 'date_balanced_mean_crps_c', 0.7599238194273233),
    (development_model_selection, {'candidate_id': 'raw_deterministic'}, 'date_balanced_mean_crps_c', 1.8493055555555555),
    (probability_evaluation_full, {'candidate_id': 'pooled_empirical_residual', 'probability_variant': 'UNCALIBRATED', 'evaluation_block': 'INTERNAL_HOLDOUT'}, 'categorical_log_score', 1.0204071791034854),
    (probability_evaluation_full, {'candidate_id': 'normalised_market', 'evaluation_block': 'INTERNAL_HOLDOUT'}, 'categorical_log_score', 1.091349772513372),
    (probability_evaluation_full, {'candidate_id': 'normalised_market', 'evaluation_block': 'EXTERNAL_TEST'}, 'categorical_log_score', 1.260922760701125),
    (probability_evaluation_full, {'candidate_id': 'gp_matern32_rule', 'probability_variant': 'CALIBRATED', 'evaluation_block': 'EXTERNAL_TEST'}, 'categorical_log_score', 1.6154967763966932),
    (trading_evaluation, {'strategy_role': 'PRIMARY_OVERALL', 'evaluation_block': 'INTERNAL_HOLDOUT'}, 'total_net_pnl', 0.2625),
    (trading_evaluation, {'strategy_role': 'PRIMARY_OVERALL', 'evaluation_block': 'EXTERNAL_TEST'}, 'total_net_pnl', -1.4925),
    (trading_evaluation, {'strategy_role': 'PRIMARY_OVERALL', 'evaluation_block': 'INTERNAL_HOLDOUT'}, 'max_drawdown', -0.5725),
    (trading_evaluation, {'strategy_role': 'PRIMARY_OVERALL', 'evaluation_block': 'EXTERNAL_TEST'}, 'max_drawdown', -1.4925),
]
for frame, filters, column, expected in expected_values:
    actual = value_at(frame, filters, column)
    if not math.isclose(actual, expected, rel_tol=0.0, abs_tol=TOL):
        raise AssertionError(f'{filters} {column}: expected {expected}, found {actual}.')

bias_min = float(deterministic_diagnostics['mean_error_c'].min())
bias_max = float(deterministic_diagnostics['mean_error_c'].max())
empirical_crps = value_at(development_model_selection, {'candidate_id': 'pooled_empirical_residual'}, 'date_balanced_mean_crps_c')
gp_crps = value_at(development_model_selection, {'candidate_id': 'gp_matern32_rule'}, 'date_balanced_mean_crps_c')
tree_crps = value_at(development_model_selection, {'candidate_id': 'catboost_quantile_pooled'}, 'date_balanced_mean_crps_c')
empirical_reduction = 100.0 * (raw_crps - empirical_crps) / raw_crps
gp_reduction = 100.0 * (raw_crps - gp_crps) / raw_crps
tree_reduction = 100.0 * (raw_crps - tree_crps) / raw_crps

holdout_emp_log = value_at(
    probability_evaluation_full,
    {'candidate_id': 'pooled_empirical_residual', 'probability_variant': 'UNCALIBRATED', 'evaluation_block': 'INTERNAL_HOLDOUT'},
    'categorical_log_score',
)
holdout_market_log = value_at(
    probability_evaluation_full,
    {'candidate_id': 'normalised_market', 'evaluation_block': 'INTERNAL_HOLDOUT'},
    'categorical_log_score',
)
external_market_log = value_at(
    probability_evaluation_full,
    {'candidate_id': 'normalised_market', 'evaluation_block': 'EXTERNAL_TEST'},
    'categorical_log_score',
)
external_best_model_log = probability_evaluation_full.loc[
    (probability_evaluation_full['method_type'].eq('WEATHER_MODEL'))
    & (probability_evaluation_full['evaluation_block'].eq('EXTERNAL_TEST')),
    'categorical_log_score',
].min()
external_best_row = probability_evaluation_full.loc[
    (probability_evaluation_full['method_type'].eq('WEATHER_MODEL'))
    & (probability_evaluation_full['evaluation_block'].eq('EXTERNAL_TEST'))
    & np.isclose(probability_evaluation_full['categorical_log_score'], external_best_model_log, atol=TOL)
].iloc[0]

primary_holdout = trading_evaluation.loc[
    trading_evaluation['strategy_role'].eq('PRIMARY_OVERALL')
    & trading_evaluation['evaluation_block'].eq('INTERNAL_HOLDOUT')
].iloc[0]
primary_external = trading_evaluation.loc[
    trading_evaluation['strategy_role'].eq('PRIMARY_OVERALL')
    & trading_evaluation['evaluation_block'].eq('EXTERNAL_TEST')
].iloc[0]

verified_claims = pd.DataFrame(
    [
        {
            'claim_id': 'C01',
            'topic': 'Certified sample',
            'status': 'VERIFIED',
            'statement': 'The final certified sample contains 103 settlement dates and 1,133 mutually exclusive temperature contracts.',
            'primary_source': str((S_DIR / '18s_expanded_sample_summary.json').relative_to(ROOT)),
            'support_rows': 1133,
            'support_dates': 103,
            'thesis_use': 'Results sample description',
        },
        {
            'claim_id': 'C02',
            'topic': 'Deterministic forecast bias',
            'status': 'VERIFIED',
            'statement': f'The deterministic forecast underpredicts HKO daily maximum temperature in the combined sample, with mean error between {bias_min:.3f} and {bias_max:.3f} degrees Celsius across the four decision rules.',
            'primary_source': str(KEY_INPUTS['weather_error'].relative_to(ROOT)),
            'support_rows': int(deterministic_diagnostics['n_complete_books'].sum()),
            'support_dates': int(s_summary['certified_dates']),
            'thesis_use': 'Forecast diagnostic result',
        },
        {
            'claim_id': 'C03',
            'topic': 'Development model selection',
            'status': 'VERIFIED',
            'statement': f'The pooled empirical residual distribution records the lowest development CRPS ({empirical_crps:.3f}), a {empirical_reduction:.1f}% reduction relative to the raw deterministic forecast.',
            'primary_source': str(KEY_INPUTS['development_scores'].relative_to(ROOT)),
            'support_rows': int(vC_summary['common_candidate_selection_rows']),
            'support_dates': int(vC_summary['common_candidate_selection_dates']),
            'thesis_use': 'Principal model-selection result',
        },
        {
            'claim_id': 'C04',
            'topic': 'Family comparison',
            'status': 'VERIFIED',
            'statement': f'The selected Matérn Gaussian process and pooled CatBoost comparators reduce development CRPS by {gp_reduction:.1f}% and {tree_reduction:.1f}% relative to the raw deterministic forecast, respectively.',
            'primary_source': str(KEY_INPUTS['development_scores'].relative_to(ROOT)),
            'support_rows': int(vC_summary['common_candidate_selection_rows']),
            'support_dates': int(vC_summary['common_candidate_selection_dates']),
            'thesis_use': 'GP and tree comparison',
        },
        {
            'claim_id': 'C05',
            'topic': 'Predictive interval calibration',
            'status': 'VERIFIED',
            'statement': 'All three selected probabilistic families under-cover the nominal 80% interval on development data; the pooled CatBoost distribution is the most concentrated.',
            'primary_source': str(KEY_INPUTS['development_scores'].relative_to(ROOT)),
            'support_rows': int(vC_summary['common_candidate_selection_rows']),
            'support_dates': int(vC_summary['common_candidate_selection_dates']),
            'thesis_use': 'Motivation for calibration',
        },
        {
            'claim_id': 'C06',
            'topic': 'Calibration transfer',
            'status': 'VERIFIED',
            'statement': 'Development-selected calibration materially repairs CatBoost zero-probability failures but does not uniformly improve holdout or June scores for the empirical and Gaussian-process models.',
            'primary_source': str(KEY_INPUTS['calibration_effect'].relative_to(ROOT)),
            'support_rows': int(wD_summary['model_book_score_rows']),
            'support_dates': 40,
            'thesis_use': 'Calibration interpretation',
        },
        {
            'claim_id': 'C07',
            'topic': 'Internal holdout comparison',
            'status': 'VERIFIED_DESCRIPTIVE',
            'statement': f'On the ten-date locked holdout, the uncalibrated empirical model has lower categorical log score than the normalised market ({holdout_emp_log:.3f} versus {holdout_market_log:.3f}); this is a descriptive comparison, not a claim of statistical significance.',
            'primary_source': str(KEY_INPUTS['probability_exact_common'].relative_to(ROOT)),
            'support_rows': int(wD_summary['exact_common_holdout_books']),
            'support_dates': 10,
            'thesis_use': 'Holdout result with caution',
        },
        {
            'claim_id': 'C08',
            'topic': 'June external comparison',
            'status': 'VERIFIED',
            'statement': f'On the thirty-date June external block, the normalised market outperforms every weather-model variant; its categorical log score is {external_market_log:.3f}, compared with {external_best_model_log:.3f} for the best model variant ({external_best_row.method_label}).',
            'primary_source': str(KEY_INPUTS['probability_exact_common'].relative_to(ROOT)),
            'support_rows': int(wD_summary['exact_common_external_books']),
            'support_dates': 30,
            'thesis_use': 'Principal external probability result',
        },
        {
            'claim_id': 'C09',
            'topic': 'Primary trading strategy',
            'status': 'VERIFIED',
            'statement': f'The development-frozen primary strategy earns net PnL {primary_holdout.total_net_pnl:.4f} on the holdout but {primary_external.total_net_pnl:.4f} in June at a hypothetical cost of 0.01 per YES share.',
            'primary_source': str(KEY_INPUTS['trading_headline'].relative_to(ROOT)),
            'support_rows': int(primary_holdout.trade_count + primary_external.trade_count),
            'support_dates': int(primary_holdout.dates + primary_external.dates),
            'thesis_use': 'Principal trading result',
        },
        {
            'claim_id': 'C10',
            'topic': 'Trading generalisation',
            'status': 'VERIFIED',
            'statement': 'Positive internal-holdout performance does not persist in June for either the empirical or Gaussian-process strategy, while the tree strategy is loss-making in both blocks.',
            'primary_source': str(KEY_INPUTS['trading_headline'].relative_to(ROOT)),
            'support_rows': int(trading_evaluation['trade_count'].sum()),
            'support_dates': 40,
            'thesis_use': 'Trading conclusion',
        },
    ]
)

evidential_boundaries = pd.DataFrame(
    [
        ('HKO outcome availability', 'All 103 dates use the analytical 14:00 HKT working-day fallback; no observed publication timestamp was supplied.', 'VERIFIED_LIMITATION', 'Do not describe the fallback as an exact observed publication time.', str((TA_DIR / '18tA_summary.json').relative_to(ROOT))),
        ('Weather source', 'The forecast source is deterministic ECMWF IFS single-run data obtained through Open-Meteo, not AIFS ENS or a genuine ensemble archive.', 'VERIFIED_LIMITATION', 'Describe it as a deterministic forecast input and not as an ensemble.', str((S_DIR / '18s_expanded_deterministic_weather_panel.csv').relative_to(ROOT))),
        ('Development selection', 'Model, calibration and trading choices are made on development information only.', 'VERIFIED_BOUNDARY', 'Holdout and June outcomes must not be described as contributing to selection.', str((VC_DIR / '18vC_selected_candidates.json').relative_to(ROOT))),
        ('Internal holdout', 'The locked holdout contains ten settlement dates.', 'VERIFIED_LIMITATION', 'Report holdout advantages descriptively and avoid significance claims.', str((UA_DIR / '18uA_summary.json').relative_to(ROOT))),
        ('External test', 'June 2026 contains thirty settlement dates and 119 model-ready date-rule rows.', 'VERIFIED_BOUNDARY', 'Treat June as the principal external out-of-time evaluation.', str((UA_DIR / '18uA_summary.json').relative_to(ROOT))),
        ('Market comparison', 'Distributional comparisons use normalised complete market books; trading uses raw observed prices.', 'VERIFIED_BOUNDARY', 'Do not mix normalised scoring probabilities with raw trading prices.', str((WD_DIR / '18wD_protocol.json').relative_to(ROOT))),
        ('Trading execution', 'The observed pre-cutoff YES price is a fill proxy; bid-ask spread, liquidity, slippage, partial fills and market impact are not modelled.', 'VERIFIED_LIMITATION', 'Do not present the simulation as an executable order-book backtest.', str((XA_DIR / '18xA_protocol.json').relative_to(ROOT))),
        ('Trading scale', 'Strategies buy one YES share only and use a primary hypothetical cost of 0.01 per trade.', 'VERIFIED_ASSUMPTION', 'State the position size and cost assumption with every headline PnL result.', str((XA_DIR / '18xA_selected_strategy_registry.csv').relative_to(ROOT))),
        ('Bootstrap', 'Date-level bootstrap intervals are descriptive for small evaluation blocks.', 'VERIFIED_LIMITATION', 'Do not interpret the bootstrap as definitive statistical evidence.', str((XD_DIR / '18xD_bootstrap_summary.csv').relative_to(ROOT))),
        ('Deferred extension', 'Genuine AIFS or ensemble forecast inputs remain outside the verified empirical release.', 'DEFERRED', 'Present ensemble and AIFS work as an extension rather than a completed result.', str((S_DIR / '18s_expanded_sample_summary.json').relative_to(ROOT))),
    ],
    columns=['boundary', 'evidence', 'status', 'required_thesis_wording', 'primary_source'],
)

check_rows: list[dict[str, Any]] = []
def add_check(check: str, passed: bool, detail: str) -> None:
    check_rows.append({'check': check, 'passed': bool(passed), 'detail': detail, 'blocking': True})

add_check('upstream_release_stages_18', len(release_registry) == 18, f'rows={len(release_registry)}')
add_check('all_upstream_verdicts_pass', release_registry['verdict'].eq('PASS').all(), '18s through 18x')
add_check('all_upstream_integrity_checks_pass', (release_registry['integrity_checks_passed'] == release_registry['integrity_checks_total']).all(), f"checks={release_registry['integrity_checks_total'].sum()}")
add_check('all_upstream_issue_files_empty', release_registry['issue_rows'].sum() == 0, 'issue rows=0')
add_check('sample_counts_frozen', all(int(s_summary[key]) == value for key, value in expected_counts.items()), json.dumps(expected_counts))
add_check('development_selection_frozen', all(development_selection[key] == value for key, value in expected_selection.items()), json.dumps(expected_selection))
add_check('probability_holdout_books_40', int(wD_summary['exact_common_holdout_books']) == 40, 'ten settlement dates, four rules')
add_check('probability_external_books_114', int(wD_summary['exact_common_external_books']) == 114, 'thirty settlement dates with five unavailable books')
add_check('trading_headline_rows_6', len(trading_evaluation) == 6, f'rows={len(trading_evaluation)}')
add_check('verified_claims_10', len(verified_claims) == 10, f'rows={len(verified_claims)}')
add_check('evidential_boundaries_10', len(evidential_boundaries) == 10, f'rows={len(evidential_boundaries)}')
add_check('model_selection_not_rerun', True, '18y reads frozen 18vC selection')
add_check('calibration_selection_not_rerun', True, '18y reads frozen 18wB parameters')
add_check('trading_selection_not_rerun', True, '18y reads frozen 18xA strategy registry')

integrity = pd.DataFrame(check_rows)
if not integrity['passed'].all():
    raise AssertionError('18yA blocking checks failed:\n' + integrity.loc[~integrity['passed']].to_string(index=False))

issues = pd.DataFrame(
    columns=['issue_level', 'issue_code', 'topic', 'detail', 'blocking']
)

print('Final empirical freeze assertions: PASS')
display(verified_claims)
display(evidential_boundaries)

Final empirical freeze assertions: PASS
  claim_id  ...                             thesis_use
0      C01  ...             Results sample description
1      C02  ...             Forecast diagnostic result
2      C03  ...       Principal model-selection result
3      C04  ...                 GP and tree comparison
4      C05  ...             Motivation for calibration
5      C06  ...             Calibration interpretation
6      C07  ...            Holdout result with caution
7      C08  ...  Principal external probability result
8      C09  ...               Principal trading result
9      C10  ...                     Trading conclusion

[10 rows x 8 columns]
                   boundary  ...                                     primary_source
0  HKO outcome availability  ...  data/processed/18tA_hko_publication_availabili...
1            Weather source  ...  data/processed/18s_expanded_march_june_canonic...
2     Development selection  ...  data/processed/18vC_common_support_scoring_and

In [5]:
output_frames = {
    'release_registry': release_registry,
    'sample_flow': final_sample_flow,
    'deterministic_diagnostics': deterministic_diagnostics,
    'market_diagnostics': market_diagnostics,
    'development_model_selection': development_model_selection,
    'calibration_summary': calibration_summary,
    'calibration_effect': calibration_effect,
    'probability_support': probability_support,
    'probability_evaluation_full': probability_evaluation_full,
    'probability_evaluation_compact': probability_evaluation_compact,
    'trading_strategy_selection': trading_strategy_selection,
    'trading_evaluation': trading_evaluation,
    'verified_claims': verified_claims,
    'evidential_boundaries': evidential_boundaries,
    'integrity': integrity,
    'issues': issues,
}

output_paths = {
    'release_registry': OUT_DIR / '18yA_release_registry.csv',
    'sample_flow': OUT_DIR / '18yA_final_sample_flow.csv',
    'deterministic_diagnostics': OUT_DIR / '18yA_deterministic_forecast_diagnostics.csv',
    'market_diagnostics': OUT_DIR / '18yA_market_baseline_diagnostics.csv',
    'development_model_selection': OUT_DIR / '18yA_development_model_selection.csv',
    'calibration_summary': OUT_DIR / '18yA_calibration_selection.csv',
    'calibration_effect': OUT_DIR / '18yA_calibration_transfer_effects.csv',
    'probability_support': OUT_DIR / '18yA_probability_evaluation_support.csv',
    'probability_evaluation_full': OUT_DIR / '18yA_probability_evaluation_full.csv',
    'probability_evaluation_compact': OUT_DIR / '18yA_probability_evaluation_compact.csv',
    'trading_strategy_selection': OUT_DIR / '18yA_trading_strategy_selection.csv',
    'trading_evaluation': OUT_DIR / '18yA_trading_evaluation.csv',
    'verified_claims': OUT_DIR / '18yA_verified_claims.csv',
    'evidential_boundaries': OUT_DIR / '18yA_evidential_boundaries.csv',
    'integrity': OUT_DIR / '18yA_integrity_checks.csv',
    'issues': OUT_DIR / '18yA_issues.csv',
}

for key, frame in output_frames.items():
    frame.to_csv(output_paths[key], index=False)

source_rows = []
for stage, directory, summary_name, _, _, manifest_name in RELEASES:
    for role, path in [
        (f'{stage}_summary', directory / summary_name),
        (f'{stage}_manifest', directory / manifest_name),
    ]:
        source_rows.append(
            {
                'input_role': role,
                'path': str(path.relative_to(ROOT)),
                'rows': 1 if path.suffix == '.json' else len(pd.read_csv(path)),
                'sha256': sha256_file(path),
            }
        )
for role, path in KEY_INPUTS.items():
    source_rows.append(
        {
            'input_role': role,
            'path': str(path.relative_to(ROOT)),
            'rows': 1 if path.suffix == '.json' else len(pd.read_csv(path)),
            'sha256': sha256_file(path),
        }
    )
source_inventory = pd.DataFrame(source_rows).drop_duplicates(['path']).reset_index(drop=True)
source_inventory_path = OUT_DIR / '18yA_source_inventory.csv'
source_inventory.to_csv(source_inventory_path, index=False)

protocol = {
    'step': STEP,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'verdict': 'PASS',
    'purpose': 'Consolidate the verified 18s–18x empirical release into final dissertation-facing evidence tables.',
    'upstream_release_stages_verified': int(len(release_registry)),
    'model_selection_rerun': False,
    'calibration_selection_rerun': False,
    'trading_selection_rerun': False,
    'development_selection_source': str(KEY_INPUTS['development_selection'].relative_to(ROOT)),
    'calibration_selection_source': str(KEY_INPUTS['calibration_parameters'].relative_to(ROOT)),
    'trading_selection_source': str(KEY_INPUTS['strategy_registry'].relative_to(ROOT)),
    'verified_claims': int(len(verified_claims)),
    'evidential_boundaries': int(len(evidential_boundaries)),
    'internal_holdout_interpretation': 'descriptive; ten settlement dates; no statistical-significance claim',
    'external_test_interpretation': 'principal out-of-time evaluation; June 2026; thirty settlement dates',
    'verified_provisional_deferred_boundary_preserved': True,
}
protocol_path = OUT_DIR / '18yA_protocol.json'
protocol_path.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding='utf-8')

key_metrics = {
    'certified_dates': int(s_summary['certified_dates']),
    'certified_contracts': int(s_summary['certified_contracts']),
    'development_dates': int(uA_summary['development_dates']),
    'common_selection_rows': int(vC_summary['common_candidate_selection_rows']),
    'holdout_dates': int(uA_summary['internal_holdout_dates']),
    'external_dates': int(uA_summary['external_test_dates']),
    'selected_baseline': development_selection['selected_baseline'],
    'selected_gaussian_process': development_selection['selected_gaussian_process'],
    'selected_tree': development_selection['selected_tree'],
    'selected_overall': development_selection['selected_overall'],
    'development_empirical_crps': empirical_crps,
    'development_raw_crps': raw_crps,
    'development_empirical_crps_reduction_pct': empirical_reduction,
    'holdout_empirical_uncalibrated_categorical_log': holdout_emp_log,
    'holdout_market_categorical_log': holdout_market_log,
    'external_market_categorical_log': external_market_log,
    'external_best_model_categorical_log': float(external_best_model_log),
    'external_best_model_method': str(external_best_row.method_label),
    'primary_holdout_net_pnl': float(primary_holdout.total_net_pnl),
    'primary_external_net_pnl': float(primary_external.total_net_pnl),
    'primary_holdout_max_drawdown': float(primary_holdout.max_drawdown),
    'primary_external_max_drawdown': float(primary_external.max_drawdown),
}

summary = {
    'step': STEP,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'verdict': 'PASS',
    'upstream_release_stages_verified': int(len(release_registry)),
    'upstream_manifest_entries_verified': int(release_registry['manifest_entries_verified'].sum()),
    'upstream_integrity_checks_verified': int(release_registry['integrity_checks_total'].sum()),
    'final_sample_flow_rows': int(len(final_sample_flow)),
    'development_model_rows': int(len(development_model_selection)),
    'probability_evaluation_full_rows': int(len(probability_evaluation_full)),
    'trading_evaluation_rows': int(len(trading_evaluation)),
    'verified_claim_rows': int(len(verified_claims)),
    'evidential_boundary_rows': int(len(evidential_boundaries)),
    'model_selection_rerun': False,
    'calibration_selection_rerun': False,
    'trading_selection_rerun': False,
    'key_metrics': key_metrics,
    'issue_rows': 0,
    'integrity_checks_passed': int(integrity['passed'].sum()),
    'integrity_checks_total': int(len(integrity)),
}
summary_path = OUT_DIR / '18yA_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')

environment = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'revision': 'v1',
}
environment_path = OUT_DIR / '18yA_environment.json'
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')

report_lines = [
    '# 18yA final empirical synthesis',
    '',
    '**PASS**',
    '',
    '## Release-chain audit',
    '',
    f"- Verified upstream release stages: {len(release_registry)}",
    f"- Verified upstream manifest entries: {release_registry['manifest_entries_verified'].sum():,}",
    f"- Verified upstream integrity checks: {release_registry['integrity_checks_total'].sum():,}",
    '- Upstream issue rows: 0',
    '',
    '## Final sample',
    '',
    f"- Certified settlement dates: {s_summary['certified_dates']:,}",
    f"- Certified contracts: {s_summary['certified_contracts']:,}",
    f"- Exact market-weather common rows: {s_summary['exact_common_support_rows']:,}",
    f"- Complete common-support books: {s_summary['complete_common_support_books']:,}",
    '',
    '## Model selection',
    '',
    f"- Overall and baseline winner: {development_selection['selected_overall']}",
    f"- Gaussian-process winner: {development_selection['selected_gaussian_process']}",
    f"- Tree winner: {development_selection['selected_tree']}",
    f"- Empirical CRPS reduction relative to raw forecast: {empirical_reduction:.1f}%",
    '',
    '## Probability evaluation',
    '',
    f"- Holdout empirical uncalibrated categorical log: {holdout_emp_log:.6f}",
    f"- Holdout normalised-market categorical log: {holdout_market_log:.6f}",
    f"- June normalised-market categorical log: {external_market_log:.6f}",
    f"- June best model categorical log: {external_best_model_log:.6f} ({external_best_row.method_label})",
    '',
    '## Trading evaluation',
    '',
    f"- Primary holdout net PnL: {primary_holdout.total_net_pnl:.4f}",
    f"- Primary June net PnL: {primary_external.total_net_pnl:.4f}",
    f"- Primary holdout maximum drawdown: {primary_holdout.max_drawdown:.4f}",
    f"- Primary June maximum drawdown: {primary_external.max_drawdown:.4f}",
    '',
    '## Evidential boundary',
    '',
    'The ten-date holdout is descriptive. June is the principal external out-of-time block. '
    'No model, calibration or trading selection is rerun in 18yA.',
]
report_path = REPORT_DIR / '18yA_final_empirical_synthesis_report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob('*')):
        if path.is_file() and path.name != '18yA_sha256_manifest.csv':
            manifest_rows.append(
                {'path': str(path.relative_to(ROOT)), 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)}
            )
manifest_path = OUT_DIR / '18yA_sha256_manifest.csv'
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

print(json.dumps(summary, indent=2))
print('18yA final empirical synthesis release: PASS')

{
  "step": "18yA",
  "generated_at_utc": "2026-07-22T18:20:29.581462+00:00",
  "verdict": "PASS",
  "upstream_release_stages_verified": 18,
  "upstream_manifest_entries_verified": 234,
  "upstream_integrity_checks_verified": 193,
  "final_sample_flow_rows": 19,
  "development_model_rows": 9,
  "probability_evaluation_full_rows": 14,
  "trading_evaluation_rows": 6,
  "verified_claim_rows": 10,
  "evidential_boundary_rows": 10,
  "model_selection_rerun": false,
  "calibration_selection_rerun": false,
  "trading_selection_rerun": false,
  "key_metrics": {
    "certified_dates": 103,
    "certified_contracts": 1133,
    "development_dates": 38,
    "common_selection_rows": 136,
    "holdout_dates": 10,
    "external_dates": 30,
    "selected_baseline": "pooled_empirical_residual",
    "selected_gaussian_process": "gp_matern32_rule",
    "selected_tree": "catboost_quantile_pooled",
    "selected_overall": "pooled_empirical_residual",
    "development_empirical_crps": 0.7112529402436527,
  